1. Install Libraries

In [1]:
## kernel - ids
"""
Persist Salesforce object field metadata (df_meta) into a SQL Server table.

- Creates the table if it doesn't exist
- Appends new rows OR performs an UPSERT (MERGE) if enabled
- Uses fast_executemany for performance
"""

#Importing Other Libraries
from simple_salesforce import Salesforce,SalesforceLogin,SalesforceAuthenticationFailed,SFType
import pandas as pd
import json
import numpy as np
import time as t
import uuid
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from collections import OrderedDict

from core_functions import *
from sf_source_objects import *
from sf_target_objects import *


In [2]:
# ----------------------------
# PARAMETERS
# 1) CONFIGURATION
# ----------------------------
p_db_server  = 'Rhombus-SQL-Ser'
# p_db  = 'ADVICE_ETL_PRD'
p_db  = 'SalesforceMigration'
p_sch = 'ids'
p_tbl_fd = 'sf_object_fd'
p_tbl_pl = 'sf_object_pl'
# p_sf_src = 'hubuat'
p_sf_src = 'hubprod'
# p_sf_tgt = 'rhpartial'
# p_sf_tgt = 'rhdata'
# p_sf_tgt = 'rhuat'
# p_sf_tgt = 'datasb'
# p_sf_tgt = 'flow2sb'
p_sf_tgt = 'rhsit'

# ----------------------------
# CONSTANTS
# WRITE: APPEND or UPSERT
# ----------------------------
UPSERT_ENABLED = True  # set False to just append/replace
WRITE_CHUNK_SIZE = 10_000

connection_string = (
    f"mssql+pyodbc://@{p_db_server}/{p_db}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&fast_executemany=True"
)


## Configure Logging
This section initializes a logging framework to monitor the ETL workflow’s execution and facilitate debugging by:
Creating an isolated logger named ETL_Salesforce.
Setting the log level to INFO to capture all important messages.
Outputting formatted messages (with timestamps and levels) to the console.
Disabling propagation to prevent duplicate log entrie

In [3]:
# Configuring logging
import logging
 
# Create a dedicated logger for the ETL process
logger = logging.getLogger('ETL_Salesforce')
logger.setLevel(logging.INFO)
 
# Clear any existing handlers to avoid duplicates
logger.handlers = []  # Remove all handlers from this logger
 
# Create a new StreamHandler with the desired format
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(handler)
 
# Disable propagation to prevent root logger interference
logger.propagate = False

## Establish Connections
This section establishes a connection to the source Salesforce environment using credentials securely fetched from Google Colab’s secrets storage.
Connecting to the Source Salesforce Instance:

In [4]:
#Connecting To The Source Salesforce Instance
from secrets_ids import secrets
url = secrets.get(p_sf_src, {}).get('site', 'Unknown')
username = secrets.get(p_sf_src, {}).get('username', 'Unknown')
pswd = secrets.get(p_sf_src, {}).get('password', 'Unknown')
token = secrets.get(p_sf_src, {}).get('token', 'Unknown')

# for PROD login
src_sf_instance = connect_to_salesforce(url=url, username=username, password=pswd, token=token, domain='login', logger=logger)
# for Sandbox login
# src_sf_instance = connect_to_salesforce(url=url, username=username, password=pswd, token=token, domain='test', logger=logger)
 
logger.info(f"Source SF Instance Session ID = {src_sf_instance.session_id}")

2026-09-04 02:22:57,808 [INFO] Connected successfully to https://ioofadviserhub.my.salesforce.com
2026-09-04 02:22:57,808 [INFO] Source SF Instance Session ID = 00D6F0000029Bzd!AQEAQESP1otgva.HJlgCGnjB2gY26Ww5hzrox.kBxtWsBVSjEoiI1qdbfhbl_ZT9iBkeKs14XHmxjUQiKm9aVJO52EzxtX6.


In [5]:
#Connecting To The Destination Salesforce Instance
from secrets_ids import secrets
url_tgt = secrets.get(p_sf_tgt, {}).get('site', 'Unknown')
username_tgt = secrets.get(p_sf_tgt, {}).get('username', 'Unknown')
pswd_tgt = secrets.get(p_sf_tgt, {}).get('password', 'Unknown')
token_tgt = secrets.get(p_sf_tgt, {}).get('token', 'Unknown')

dst_sf_instance = connect_to_salesforce(url=url_tgt, username=username_tgt, password=pswd_tgt, token=token_tgt, domain='test', logger=logger)
 
logger.info(f"Destination SF Instance Session ID = {dst_sf_instance.session_id}")

2026-09-04 02:23:00,398 [INFO] Connected successfully to https://test.salesforce.com
2026-09-04 02:23:00,399 [INFO] Destination SF Instance Session ID = 00DBm00000CWPlx!AQEAQJisfayl5n7.FwzSKP.ctprm4b84oZySLRyBaIa8MAzE8jvHcOxYPPc05qLr3sPwjfiGuz76lF5JCcIqb2.XZT8KKTbS


To maintain security, credentials are not hardcoded. Instead, they are stored in Google Colab’s Secrets vault. The application dynamically accesses these stored values (src_username, src_password, and src_token) via the userdata.get('secret_name') method. The fetched credentials are passed to the connect_to_salesforce function to authenticate and create a Salesforce client instance.

Connecting to the Destination Salesforce Instance:

## Data Extraction Phase
This section handles the data extraction process by:
Defining the Salesforce object to query (Account) and the key reference column (Name) used for data validation and uniqueness checks.
Constructing a SOQL query string to retrieve the Id, Name, and Phone fields while limiting the results to three records to get a small sample of data for testing.
Calling the extract_query function with the Salesforce connection and SOQL query; This function executes the query and loads the results into a panda DataFrame (object_df).
Logging the count of non-null values per column to identify missing data.
Logging the total number of records extracted.
Logging the count of unique values in the reference column (Name).
Checking for duplicates in the reference column and logging a warning if any are found.
Displaying the first five rows of the extracted DataFrame to give a preview of the data structure and content.

In [ ]:
from sf_source_objects import SOURCE_OBJECTS

ls_objects = SOURCE_OBJECTS

## Extract Meta data of objects from src SF
Input - ls_objects - holds a list of object names identified at the src org.
Note that due to the number of objects exceeds 2 hundred, the script below should be tested and validated with a few objects before running it over all objects.

In [6]:
## Extract from IFL instance
## To test the script or to re-extract the metadata of one object, include only the object in the list below.
# ls_objects = SOURCE_OBJECTS
ls_objects = [
    # 'Adviser_ASN_Snapshot__c'
    # 'ASF_CSA_Case__c'
    # ,'CSA_Case__c'
    # 'User'
    # 'PermissionSetAssignment'
    # ,'PermissionSet'
    # 'Contact'
    # ,'PermissionSetLicense'
    # ,'UserLicense'
    'Attachment'
]

for objectName in ls_objects: 
    print(f"Processing {objectName}.")
    df_meta = extract_sfObject_metadata(src_sf_instance,objectName)
    df_meta = normalize_df(df_meta)

    # Merge into table
    engine = get_engine(connection_string)
    with engine.begin() as conn:
        conn.execute(text('TRUNCATE TABLE [SalesforceMigration].[ids].[stg_fd]'))
        conn.commit()

    df_meta.to_sql(
        name='stg_fd',
        con=get_engine(connection_string),
        schema='ids',
        if_exists="append",
        index=False,
        chunksize=WRITE_CHUNK_SIZE,
        method=None
    )

    p_sch = 'ids'
    p_tbl_fd = 'sf_object_meta'

    q_sql_merge_meta = f"""
    MERGE [SalesforceMigration].[{p_sch}].[{p_tbl_fd}] AS tgt
    USING [SalesforceMigration].[{p_sch}].[stg_fd] AS src
    ON  tgt.[object_api_name] = src.[object_api_name]
    AND tgt.[field_api_name]  = src.[field_api_name]
    WHEN MATCHED THEN UPDATE SET
        tgt.[field_label]         = src.[field_label]
        ,tgt.[data_type]          = src.[data_type]
        ,tgt.[length]             = src.[length]
        ,tgt.[description]        = src.[description] 
        ,tgt.[precision]          = src.[precision]
        ,tgt.[scale]              = src.[scale]
        ,tgt.[nillable]           = src.[nillable]
        ,tgt.[unique]             = src.[unique]
        ,tgt.[calculated]         = src.[calculated]
        ,tgt.[calculated_formula] = src.[calculated_formula]
        ,tgt.[inline_help_text]   = src.[inline_help_text]
        ,tgt.[reference_to]       = src.[reference_to]
        ,tgt.[picklist_values]    = src.[picklist_values]
    WHEN NOT MATCHED BY TARGET THEN
        INSERT (
            [object_api_name], [field_api_name], [field_label], [data_type], [length], [description],
            [precision], [scale], [nillable], [unique], [calculated], [calculated_formula], [inline_help_text],
            [reference_to]
        )
        VALUES (
            src.[object_api_name], src.[field_api_name], src.[field_label], src.[data_type], src.[length], src.[description],
            src.[precision], src.[scale], src.[nillable], src.[unique], src.[calculated], src.[calculated_formula], src.[inline_help_text],
            src.[reference_to]
        )
    ;
    """
    # Merge into table
    engine = get_engine(connection_string)
    with engine.begin() as conn:
        conn.execute(text(q_sql_merge_meta))
        conn.commit()


Processing Attachment.
2026-09-04 02:23:45,524 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-04 02:23:45,526 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-04 02:23:45,528 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-04 02:23:45,529 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-04 02:23:45,532 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-04 02:23:45,536 INFO sqlalchemy.engine.Engine [generated in 0.00116s] ()
2026-09-04 02:23:45,538 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-04 02:23:45,538 INFO sqlalchemy.engine.Engine [generated in 0.00077s] ()
2026-09-04 02:23:45,540 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:23:45,541 INFO sqlalchemy.engine.Engine TRUNCATE TABLE [SalesforceMigration].[ids].[stg_fd]
2026-09-04 02:23:45,542 INFO sqlalchemy.engine

In [9]:
from sf_target_objects import TARGET_OBJECTS

ls_objects_RA = TARGET_OBJECTS

In [7]:
## Extract from Target instance (RA) for the new Salesforce Ids
# ls_objects_RA = ['User','Account','Contact','Profile','RecordType']
ls_objects_RA = [
    # 'ASF_CSA_Case__c'
    # ,'CSA_Case__c'
    # 'User'
    # ,'Account'
    # 'Contact'
    'Attachment'
    # 'Adviser_ASN_Snapshot__c'
    # 'Refund_Request__c'
    # 'PermissionSetAssignment'
    # ,'PermissionSet'
    # ,'PermissionSetGroup'
    # ,'PermissionSetLicense'
    # ,'UserLicense'
    ]

## RA instance
for objectName in ls_objects_RA: 
    print(f"Processing {objectName}.")

    try:
        df_meta = extract_sfObject_metadata(dst_sf_instance,objectName)
        df_meta = normalize_df(df_meta)

        # Merge into table
        engine = get_engine(connection_string)
        with engine.begin() as conn:
            conn.execute(text('TRUNCATE TABLE [ids].[stg_fd_ra]'))
            conn.commit()

        df_meta.to_sql(
            name='stg_fd_ra',
            con=get_engine(connection_string),
            schema='ids',
            if_exists="append",
            index=False,
            chunksize=WRITE_CHUNK_SIZE,
            method=None
        )

        p_sch = 'ids'
        p_tbl_fd = 'sf_object_meta_ra'

        q_sql_merge_meta_ra = f"""
        MERGE [SalesforceMigration].[{p_sch}].[{p_tbl_fd}] AS tgt
        USING [SalesforceMigration].[{p_sch}].[stg_fd_ra] AS src
        ON  tgt.[object_api_name] = src.[object_api_name]
        AND tgt.[field_api_name]  = src.[field_api_name]
        WHEN MATCHED THEN UPDATE SET
            tgt.[field_label]         = src.[field_label]
            ,tgt.[data_type]          = src.[data_type]
            ,tgt.[length]             = src.[length]
            ,tgt.[description]        = src.[description] 
            ,tgt.[precision]          = src.[precision]
            ,tgt.[scale]              = src.[scale]
            ,tgt.[nillable]           = src.[nillable]
            ,tgt.[unique]             = src.[unique]
            ,tgt.[calculated]         = src.[calculated]
            ,tgt.[calculated_formula] = src.[calculated_formula]
            ,tgt.[inline_help_text]   = src.[inline_help_text]
            ,tgt.[reference_to]       = src.[reference_to]
            ,tgt.[picklist_values]    = src.[picklist_values]
        WHEN NOT MATCHED BY TARGET THEN
            INSERT (
                [object_api_name], [field_api_name], [field_label], [data_type], [length], [description],
                [precision], [scale], [nillable], [unique], [calculated], [calculated_formula], [inline_help_text],
                [reference_to]
            )
            VALUES (
                src.[object_api_name], src.[field_api_name], src.[field_label], src.[data_type], src.[length], src.[description],
                src.[precision], src.[scale], src.[nillable], src.[unique], src.[calculated], src.[calculated_formula], src.[inline_help_text],
                src.[reference_to]
            )
        ;
        """
        # Merge into table
        engine = get_engine(connection_string)
        with engine.begin() as conn:
            conn.execute(text(q_sql_merge_meta_ra))
            conn.commit()
    except Exception as e:
        print(f"Error processing {objectName}: {str(e)}")


Processing Attachment.
2026-09-04 02:24:12,935 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-04 02:24:12,935 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-04 02:24:12,946 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-04 02:24:12,948 INFO sqlalchemy.engine.Engine [generated in 0.00152s] ()
2026-09-04 02:24:12,950 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-04 02:24:12,951 INFO sqlalchemy.engine.Engine [generated in 0.00121s] ()
2026-09-04 02:24:12,953 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-04 02:24:12,954 INFO sqlalchemy.engine.Engine [generated in 0.00061s] ()
2026-09-04 02:24:12,956 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:24:12,958 INFO sqlalchemy.engine.Engine TRUNCATE TABLE [ids].[stg_fd_ra]
2026-09-04 02:24:12,958 INFO sqlalchemy.engine.Engine [generated 